In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
p450plant0 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant0.pkl")
)
p450plant1 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant1.pkl")
)
p450plant2 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant2.pkl")
)
p450plant3 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant3.pkl")
)
p450plant4 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant4.pkl")
)


p450plant = pd.concat(
    [p450plant0, p450plant1, p450plant2, p450plant3, p450plant4], ignore_index=True
)

In [3]:
p450plant

,enzyme,substrate,Binding,ECFP,ESM1b
0,CYP71AY5,GEI,1,0000000000000000000000000000000001000000000010...,"[-0.032707780599594116, 0.15956459939479828, -..."
1,CYP85A2,CAT,1,0100000010000000000000000000010001001000000000...,"[-0.12464731186628342, 0.25692933797836304, -0..."
2,CYP716A94,BAM,1,0000000000000000000000000000000101001000000000...,"[-0.1793026179075241, 0.2773324251174927, 0.02..."
3,CYP80B2,NME,1,0001000000000000000000000100000001000000100000...,"[-0.05789301171898842, 0.1279979795217514, -0...."
4,CYP85A69,DOC,1,0100000010000000000000000000000001001000000000...,"[-0.077804334461689, 0.2830597162246704, -0.00..."
...,...,...,...,...,...
1546,CYP71D495,LUP,0,0000000000000000000000000000000101001000000000...,"[-0.08674226701259613, 0.1240856796503067, -0...."
1547,CYP716A155,Vincadifformine,0,0000100000000000000000000000000001001000000000...,"[-0.16968412697315216, 0.2657066881656647, 0.0..."
1548,CYP716A155,KEO,0,0000100000000000000000000000000001011000000000...,"[-0.16968412697315216, 0.2657066881656647, 0.0..."
1549,CYP706C55,LIT,0,0100000000000000000000000010000001000000011000...,"[-0.07378055155277252, 0.1300899088382721, 0.0..."


In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

data_X, data_y = create_input_and_output_data(df=p450plant)

In [ ]:
bst = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "p450authormodel.dat")
)
dtest_new = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)

In [ ]:
y_test_new_pred = np.round(bst.predict(dtest_new))
acc_test_new = np.mean(y_test_new_pred == np.array(data_y))
try:
    roc_auc_new = roc_auc_score(np.array(data_y), bst.predict(dtest_new))
    mcc = matthews_corrcoef(np.array(data_y), y_test_new_pred)
except:
    roc_auc_new = 0
    mcc = 0

print(
    "Accuracy on test set: %s, ROC-AUC score for test set: %s, MCC: %s"
    % (acc_test_new, roc_auc_new, mcc)
)

Accuracy on test set: 0.6002578981302386, ROC-AUC score for test set: 0.5705678123678865, MCC: 0.08030594365292808


In [ ]:
index_of_ones = np.where(np.array(data_y) == 1)[0]
values_of_ones = bst.predict(dtest_new)[index_of_ones]
acc_1 = np.mean(np.round(values_of_ones) == 1)

index_of_zeros = np.where(np.array(data_y) == 0)[0]
values_of_zeros = bst.predict(dtest_new)[index_of_zeros]
acc_0 = np.mean(np.round(values_of_zeros) == 0)

print("Accuracy on 1 set: %s, Accuracy on 0 set: %s" % (acc_1, acc_0))
print(len(data_y))

Accuracy on 1 set: 0.35589941972920697, Accuracy on 0 set: 0.7224371373307543
1551


In [8]:
data_y.count(0)

1034

In [9]:
data_y.count(1)

517

In [10]:
np.array(data_y)

array([1, 1, 1, ..., 0, 0, 0])

In [ ]:
np.save(join(our_data + "notrain_y_test_pred.npy"), bst.predict(dtest_new))
np.save(join(our_data + "notrain_y_test.npy"), np.array(data_y))